# Future Interns - Data Science & Analytics
## Task 2: Customer Retention & Churn Analysis
**Author:** Data Science Intern  
**Repository:** `FUTURE_DS_02`  
**Objective:** Analyze subscription customer data to evaluate churn rates, cohort survival patterns, key attrition drivers, and build actionable retention strategies.

### 1. Import Libraries & Set Up Analysis Environment

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment initialized successfully!")

### 2. Ingest and Inspect Customer Subscription Dataset

In [2]:
data_path = os.path.join("..", "data", "cleaned_customer_churn_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]:,} customers across {df.shape[1]} features")
display(df.head(3))

### 3. Executive Retention & Churn Metrics

In [3]:
total_cust = len(df)
churned_cust = df["Churn_Binary"].sum()
churn_rate = (churned_cust / total_cust) * 100
retention_rate = 100 - churn_rate
total_mrr = df["MonthlyCharges"].sum()
lost_mrr = df[df["Churn"] == "Yes"]["MonthlyCharges"].sum()
avg_tenure = df["Tenure_Months"].mean()
avg_clv = df["Customer_Lifetime_Value"].mean()

print("=" * 50)
print("       EXECUTIVE RETENTION SCORECARD")
print("=" * 50)
print(f"Total Accounts Audited    : {total_cust:,}")
print(f"Retained Active Accounts  : {total_cust - churned_cust:,} ({retention_rate:.1f}%)")
print(f"Churned Accounts (Lost)   : {churned_cust:,} ({churn_rate:.1f}%)")
print(f"Total Monthly Revenue MRR : ${total_mrr:,.2f}")
print(f"Monthly Revenue Lost      : ${lost_mrr:,.2f} / month")
print(f"Average Account Tenure    : {avg_tenure:.1f} months")
print(f"Average Customer Lifetime : ${avg_clv:,.2f}")
print("=" * 50)

### 4. Churn Risk by Contract Commitment Structure

In [4]:
contract_summary = df.groupby("Contract_Type").agg({
    "CustomerID": "count",
    "Churn_Binary": ["sum", "mean"],
    "MonthlyCharges": "mean"
})
contract_summary.columns = ["Total_Accounts", "Churned_Accounts", "Churn_Rate", "Avg_Monthly_Fee"]
contract_summary["Churn_Rate"] = (contract_summary["Churn_Rate"] * 100).round(2)
display(contract_summary)

plt.figure(figsize=(9, 5))
sns.barplot(x=contract_summary.index, y=contract_summary["Churn_Rate"], palette="Reds_r")
plt.title("Customer Churn Rate by Contract Type (%)", fontsize=14, fontweight="bold")
plt.ylabel("Churn Rate (%)")
plt.show()

### 5. Tenure Cohort Survival & Hazard Rates

In [5]:
cohort_data = df.groupby("Tenure_Bracket", observed=False)["Churn_Binary"].agg(["count", "mean"]).reset_index()
cohort_data.columns = ["Tenure_Cohort", "Customer_Count", "Churn_Rate"]
cohort_data["Churn_Rate"] = cohort_data["Churn_Rate"] * 100
display(cohort_data)

plt.figure(figsize=(10, 5))
plt.plot(cohort_data["Tenure_Cohort"], cohort_data["Churn_Rate"], marker="o", linewidth=2.5, color="#DC2626")
plt.title("Attrition Hazard Rate Across Customer Tenure Stages", fontsize=14, fontweight="bold")
plt.ylabel("Cohort Churn Rate (%)")
plt.xlabel("Tenure Window")
plt.show()

### 6. Impact of Value-Added Support Services

In [6]:
support_churn = df.groupby("TechSupport")["Churn_Binary"].mean() * 100
security_churn = df.groupby("OnlineSecurity")["Churn_Binary"].mean() * 100

print("Churn by Tech Support Status (%):")
print(support_churn)
print("\nChurn by Online Security Status (%):")
print(security_churn)

### 7. Strategic Conclusions
1. **Contract Locking:** Shift Month-to-Month customers to Annual plans with introductory discounts.
2. **Critical 0-6 Month Onboarding:** Focus retention touchpoints within the first 90 days to bypass the cliff.
3. **Add-on Bundling:** Free Tech Support decreases churn by over 50%.